In [1]:
# ==========================================
# Step 1: Environment Setup & Dataset Ingestion
# ==========================================

# 1. ติดตั้งไลบรารีเสริม
!pip install -q kagglehub scikit-learn seaborn matplotlib torchsummary

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import kagglehub

# 2. ล็อกค่า Random Seed เพื่อให้ผลลัพธ์การทดลองทำซ้ำได้ (Reproducible)
SEED = 101
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. ดาวน์โหลดชุดข้อมูล Cars Body Type Cropped จาก Kaggle
dataset_path = kagglehub.dataset_download("ademboukhris/cars-body-type-cropped")
print(f"Dataset downloaded to: {dataset_path}")

Using device: cuda


100%|██████████| 1.34G/1.34G [00:19<00:00, 75.7MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/ademboukhris/cars-body-type-cropped/versions/1


In [4]:
# ==========================================
# Step 2: Dataset Exploration & Data Splitting (70/15/15)
# ==========================================

# 1. ค้นหาโฟลเดอร์ย่อย train, valid, test และรวบรวมไฟล์ภาพทั้งหมด
subdirs = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
base_dir = dataset_path if any(f in subdirs for f in ['train', 'valid', 'test']) else os.path.join(dataset_path, subdirs[0])

all_samples = []
class_names = []

for split_folder in ['train', 'valid', 'test']:
    folder_path = os.path.join(base_dir, split_folder)
    if os.path.exists(folder_path):
        temp_ds = datasets.ImageFolder(root=folder_path)
        all_samples.extend(temp_ds.samples)
        class_names = temp_ds.classes

num_classes = len(class_names)
total_len = len(all_samples)

print(f"Total images found: {total_len:,} images")
print(f"Detected {num_classes} Car Classes: {class_names}")

# 2. แบ่งข้อมูล 70/15/15 ด้วย Seed 101 เพื่อความสามารถในการทำซ้ำ
indices = list(range(total_len))
random.shuffle(indices)

train_split = int(0.70 * total_len)
val_split = int(0.85 * total_len)

train_samples = [all_samples[i] for i in indices[:train_split]]
val_samples = [all_samples[i] for i in indices[train_split:val_split]]
test_samples = [all_samples[i] for i in indices[val_split:]]

print(f"\nData Split Summary (Seed {SEED}):")
print(f"- Train set:      {len(train_samples):,} samples ({len(train_samples)/total_len*100:.1f}%)")
print(f"- Validation set: {len(val_samples):,} samples ({len(val_samples)/total_len*100:.1f}%)")
print(f"- Test set:       {len(test_samples):,} samples ({len(test_samples)/total_len*100:.1f}%)")

# 3. กำหนด Custom Dataset Class สำหรับโหลดภาพและรองรับ Data Transforms
class CarDataset(torch.utils.data.Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

Total images found: 7,549 images
Detected 7 Car Classes: ['Convertible', 'Coupe', 'Hatchback', 'Pick-Up', 'SUV', 'Sedan', 'VAN']

Data Split Summary (Seed 101):
- Train set:      5,284 samples (70.0%)
- Validation set: 1,132 samples (15.0%)
- Test set:       1,133 samples (15.0%)


In [5]:
# ==========================================
# Step 3: Data Transforms & DataLoaders (Leakage-Free Pipeline)
# ==========================================

IMG_SIZE = 128
BATCH_SIZE = 32

# 1. Pipeline สำหรับชุดฝึกสอน (Train): มี Data Augmentation เสริมความแข็งแกร่ง
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),          # รถหันซ้าย/ขวาเป็นประเภทเดิมได้
    transforms.RandomRotation(degrees=10),           # จำลองมุมมองกล้องเอียงเล็กน้อย
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # จำลองสภาพแสงแดดและร่มเงา
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Pipeline สำหรับชุดตรวจสอบและทดสอบ (Validation/Test): ไม่มี Augmentation
eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. ประกอบ Dataset โดยใช้ Transform ที่ถูกต้อง
train_dataset = CarDataset(train_samples, transform=train_transforms)
val_dataset   = CarDataset(val_samples,   transform=eval_transforms)
test_dataset  = CarDataset(test_samples,  transform=eval_transforms)

# 4. สร้าง DataLoader สำหรับป้อนข้อมูลเข้าโมเดล
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# 5. ทดสอบดึง 1 Batch เพื่อตรวจสอบมิติข้อมูล (Shape Check)
sample_images, sample_labels = next(iter(train_loader))
print("DataLoaders created successfully:")
print(f"- Batch shape: {sample_images.shape} (Batch, Channel, Height, Width)")
print(f"- Label shape: {sample_labels.shape}")
print(f"- Total Batches: Train={len(train_loader)}, Val={len(val_loader)}, Test={len(test_loader)}")

DataLoaders created successfully:
- Batch shape: torch.Size([32, 3, 128, 128]) (Batch, Channel, Height, Width)
- Label shape: torch.Size([32])
- Total Batches: Train=166, Val=36, Test=36
